# Demostración del proxy preventivo (respaldo en Colab)

Levanta `proxy/servidor.py` con el modelo final de tu Drive y muestra la página de demostración aquí mismo. Úsalo si la demostración en tu PC no funciona. Entorno: GPU T4.

## 1. Subir el código
Sube `modelo-ner-demo.zip`.

In [ ]:
from google.colab import files
import zipfile, os
subido = files.upload()
zipfile.ZipFile(next(iter(subido))).extractall('/content')
os.chdir('/content/modelo-ner')

## 2. Modelo final y dependencias
Si quieres que la demostración también envíe el texto a la IA, pega tu clave de OpenAI cuando la pida; si no, deja el campo vacío y presiona Enter.

In [ ]:
!pip install -q fastapi uvicorn httpx
from google.colab import drive; drive.mount('/content/drive')
MODELO = '/content/drive/MyDrive/modelos-ner/ner-bert-final'
assert os.path.exists(MODELO + '/config.json'), 'No encuentro ner-bert-final en MyDrive/modelos-ner'
from getpass import getpass
clave = getpass('Clave de OpenAI (opcional): ')
if clave: os.environ['OPENAI_API_KEY'] = clave
os.environ['NER_BACKEND'] = 'hf'; os.environ['NER_MODELO'] = MODELO

## 3. Levantar el proxy y abrir la demostración

In [ ]:
import subprocess, time, httpx
proc = subprocess.Popen(['python', '-m', 'uvicorn', 'proxy.servidor:app', '--port', '8000'], env=os.environ.copy())
for _ in range(120):
    try:
        if httpx.get('http://localhost:8000/').status_code == 200: break
    except Exception: pass
    time.sleep(1)
print('Proxy listo')
from google.colab import output
output.serve_kernel_port_as_iframe(8000, path='/', height=900)

Para abrirla en una pestaña aparte:

In [ ]:
output.serve_kernel_port_as_window(8000, path='/')